# Chapter 10. 파이썬 오류와 예외 처리

이 노트북은 [Chapter 10 원본 문서](../doc/Chapter%2010.%20%ED%8C%8C%EC%9D%B4%EC%8D%AC%20%EC%98%A4%EB%A5%98%EC%99%80%20%EC%98%88%EC%99%B8%20%EC%B2%98%EB%A6%AC.md)를 실습용으로 변환한 자료입니다.

문법 오류와 예외의 차이, `try`/`except`, `else`, `finally`, 사용자 정의 예외, `raise`, `ExceptionGroup`까지 코인 거래 예제로 연습합니다.

## 1. 실습 환경 준비

임시 디렉터리를 만들고 예외 처리 실습 결과를 안전하게 보관합니다.

In [1]:
from pathlib import Path
import tempfile

work_dir = Path(tempfile.mkdtemp(prefix="chapter10_"))
print("실습 디렉터리:", work_dir)

실습 디렉터리: C:\Users\juyoung\AppData\Local\Temp\chapter10_yo_7a3i7


## 2. 문법 오류와 예외의 차이

문법 오류는 코드의 구조가 잘못된 경우고, 예외는 문법은 맞지만 실행 중 발생하는 문제입니다.

In [ ]:
# 문법 오류 예시: 주석 처리로 보관
# if price > 100  # SyntaxError: ':' 누락
#     print(price)

for label in ["ZeroDivisionError", "TypeError"]:
    try:
        if label == "ZeroDivisionError":
            print(10 / 0)
        else:
            print("100" + 100)
    except Exception as error:
        print(f"{label}: {type(error).__name__} - {error}")

TypeError: can only concatenate str (not "int") to str

## 3. `try`와 `except`

예외가 발생할 수 있는 코드를 `try` 안에 두고, 문제를 처리하는 코드를 `except` 안에 넣습니다.

In [ ]:
text = "105000000"

try:
    price = int(text)
except ValueError:
    print("가격은 정수로 입력해야 합니다.")
else:
    print(f"가격: {price:,}원")

try:
    not_a_number = int("one hundred")
except ValueError as error:
    print("입력값 변환 실패:", error)
    print("오류 종류:", type(error).__name__)

## 4. 여러 예외 처리와 예외 정보 확인

예외 종류별로 다른 대응을 할 수 있고, 예외 객체를 통해 더 자세한 정보를 볼 수 있습니다.

In [ ]:
prices = {"BTC": 105_000_000, "ETH": 3_500_000}

def get_price(data, symbol):
    try:
        return data[symbol]
    except KeyError:
        print(f"지원하지 않는 코인입니다: {symbol}")
    except TypeError:
        print("가격 데이터는 딕셔너리여야 합니다.")
    return None

print(get_price(prices, "BTC"))
print(get_price(prices, "DOGE"))

try:
    price = int("not a number")
except (TypeError, ValueError) as error:
    print("숫자로 변환할 수 없습니다:", error)

## 5. `else`와 `finally`

`else`는 예외가 발생하지 않았을 때만 실행되고, `finally`는 마지막에 반드시 실행됩니다.

In [ ]:
def read_price(text):
    try:
        price = float(text)
    except ValueError:
        print("가격 형식이 올바르지 않습니다.")
        return None
    else:
        print("가격 변환에 성공했습니다.")
        return price

result = read_price("105.5")
print("결과:", result)


def process_order():
    print("주문 처리를 시작합니다.")
    try:
        raise RuntimeError("거래소 응답 오류")
    except RuntimeError as error:
        print(f"주문 실패: {error}")
    finally:
        print("주문 처리 자원을 정리합니다.")

process_order()

## 6. 파일과 `with`

파일을 다룰 때는 `with` 문을 사용해 예외 발생 시에도 파일이 닫히도록 만들 수 있습니다.

In [ ]:
price_path = work_dir / "prices.txt"

try:
    with open(price_path, "w", encoding="utf-8") as file:
        file.write("BTC,105000000\n")
        file.write("ETH,3500000\n")
    with open(price_path, encoding="utf-8") as file:
        content = file.read()
    print("가격 파일 내용:\n", content)
except FileNotFoundError:
    print("가격 파일을 찾을 수 없습니다.")

## 7. `raise`로 예외 발생시키기

함수는 잘못된 입력을 조용히 무시하지 않고, 적절한 예외를 직접 발생시키는 것이 좋습니다.

In [ ]:
def calculate_return(buy_price, sell_price):
    if buy_price <= 0:
        raise ValueError("매수 가격은 0보다 커야 합니다.")
    return (sell_price - buy_price) / buy_price

try:
    print(calculate_return(0, 110))
except ValueError as error:
    print(f"계산할 수 없습니다: {error}")


def load_price(path):
    try:
        with open(path, encoding="utf-8") as file:
            return float(file.read())
    except (OSError, ValueError) as error:
        print(f"가격 파일 처리 실패: {error}")
        raise

# 실제 파일이 없으면 예외가 전파됩니다.
try:
    load_price(work_dir / "missing_price.txt")
except FileNotFoundError as error:
    print("다시 발생한 예외:", error)

## 8. 예외 연결하기

원래 원인을 보존하면서 더 높은 수준의 예외로 바꾸는 예시입니다.

In [ ]:
def load_market_price(path):
    try:
        with open(path, encoding="utf-8") as file:
            return float(file.read())
    except (OSError, ValueError) as error:
        raise RuntimeError("시장 가격을 불러오지 못했습니다.") from error

try:
    load_market_price(work_dir / "missing_market_price.txt")
except RuntimeError as error:
    print(error)
    print("원인 예외:", type(error.__cause__).__name__ if error.__cause__ else "원인 없음")

## 9. 사용자 정의 예외

거래 도메인에 맞는 예외를 만들면 호출자가 일반 오류와 비즈니스 오류를 구분하기 쉽습니다.

In [ ]:
class TradingError(Exception):
    """거래 처리 중 발생하는 기본 예외입니다."""


class InvalidOrderError(TradingError):
    """주문 정보가 잘못되었을 때 발생합니다."""


class InsufficientBalanceError(TradingError):
    """잔액이 부족할 때 발생합니다."""


def place_order(balance, amount):
    if amount <= 0:
        raise InvalidOrderError("주문 금액은 0보다 커야 합니다.")
    if amount > balance:
        raise InsufficientBalanceError("잔액이 부족합니다.")
    return "주문이 접수되었습니다."

for case in [(100_000, 50_000), (10_000, 20_000), (10_000, 0)]:
    try:
        print(place_order(*case))
    except InsufficientBalanceError as error:
        print(f"잔액 오류: {error}")
    except InvalidOrderError as error:
        print(f"주문 오류: {error}")

## 10. `ExceptionGroup`과 `except*`

Python 3.11 이상에서는 여러 예외를 묶어 처리할 수 있습니다.

In [ ]:
try:
    raise ExceptionGroup(
        "주문 검증 실패",
        [ValueError("가격 오류"), TypeError("수량 형식 오류")],
    )
except* ValueError as error_group:
    print("가격 관련 오류를 처리했습니다.", len(error_group.exceptions))
except* TypeError as error_group:
    print("수량 형식 오류를 처리했습니다.", len(error_group.exceptions))

## 11. 예외 노트 추가하기

`add_note()`를 사용하면 예외에 추가 정보를 남길 수 있습니다.

In [ ]:
try:
    raise ValueError("가격을 숫자로 변환할 수 없습니다.")
except ValueError as error:
    error.add_note("파일: markets.json")
    error.add_note("항목: BTC.price")
    print("예외 메모:", error.__notes__)

## 12. 실습 검증

최종적으로 예외 처리 코드를 실행하면서 기대한 동작이 실제로 발생하는지 확인합니다.

In [ ]:
workspace_dir = Path.cwd()
while workspace_dir != workspace_dir.parent and not (workspace_dir / "doc").exists():
    workspace_dir = workspace_dir.parent
source_path = workspace_dir / "doc" / "Chapter 10. 파이썬 오류와 예외 처리.md"
assert source_path.exists(), source_path
assert price_path.exists()
assert read_price("105.5") == 105.5
print("Chapter 10 실습 검증 통과")
print("원본 문서:", source_path)
print("생성 파일:", sorted(path.name for path in work_dir.iterdir()))

## 실습 과제

1. 입력값이 비어 있거나 숫자가 아니면 사용자에게 다시 입력을 요청하는 예외 처리를 작성하세요.
2. `with open()` 예외 상황에서 파일이 실제로 닫히는지 확인하세요.
3. 자신의 거래 도메인에 맞는 사용자 정의 예외를 2개 이상 정의하고 활용하세요.
4. `raise ... from ...`을 사용해 외부 API 응답 오류를 내부 거래 오류로 변환해 보세요.
5. `ExceptionGroup`을 활용해 여러 시장 검증 오류를 한 번에 보고하는 코드를 작성하세요.